In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from pathlib import Path
import os
from os.path import join
import sys

# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully
EU1_PROD_Conn created successfully
EU2_PROD_Conn created successfully


In [3]:
from datetime import datetime, timedelta

def days_in_week_range(week_number, year):
    """
    Helper to count days in a week. If it's the current week, return days up to today.
    week_number: int week number (ISO, 1-53).
    year: int year.
    current_week: int, current week number.
    """

    # Get Monday of the ISO week
    try:
        start_date = date.fromisocalendar(int(year), int(week_number), 1)
    except Exception:
        # If not a valid ISO week/year, fall back to 7
        return 7

    # End on Sunday
    end_date = start_date + timedelta(days=6)

    today = datetime.now().date()
    this_iso = today.isocalendar()[:2]  # (year, week)

    if (int(year), int(week_number)) == this_iso:
        delta = (today - start_date).days + 1  # include today
        return min(max(delta, 0), 7)
    else:
        return 7

In [4]:
customer_name = 'Cadent'
customer_id = Query(query = f"SELECT CustomerId FROM KPI_Customer WHERE Name = '{customer_name}'").execute([KPIHub_Conn]).values[0][0]

customer_utilization = {'Hours':7, 'Days':7}
tableList = [KPI_ReportSummary,KPI_SurveySummary]
aggregator = ['ReportYear', 'ReportWeek']
data = {}
for table in tableList:
    data[table] = Query(query = f"SELECT * FROM {table.name} WHERE ReportId IN (SELECT ReportId FROM KPI_ReportSummary WHERE CustomerId IN (SELECT CustomerId FROM KPI_Customer WHERE Name = '{customer_name}'))").execute([KPIHub_Conn])

def report_KPI_apply(df):
    if hasattr(df, 'name') and df.name is not None:
        group_year = df.name[0]
        group_week_range = df.name[1]
    else:
        group_year, group_week_range = None, None
    current_year = datetime.now().year

    current_week = datetime.now().isocalendar()[1]
    if group_year is not None and group_week_range is not None:
        day_count = days_in_week_range(group_week_range, current_year)
    else:
        day_count = None
    
    unique_reports = df.drop_duplicates(subset=['ReportId'])
    no_surveyors = df['SurveyorUnit'].nunique()
    surveyDurationHours = df['SurveyDurationMinutes'].sum()/60
    targetTimeHours = day_count*customer_utilization['Hours']*no_surveyors
    starndardTargetTimeHours = 6*5*25
    #targetTimeHours = 7*7*25
    surveyCount = df['SurveyId'].nunique()
    avg_speed_weighted = df['AvgSpeedKm'] * df['TotalSegments']
    print(df.name, no_surveyors, surveyCount)
    return pd.Series({
        'SurveyDurationHours': surveyDurationHours,
        'TargetDurationHours': targetTimeHours,
        'CustomerUtilization': surveyDurationHours/targetTimeHours,
        'StarndardUtilization': surveyDurationHours/starndardTargetTimeHours,
        'TotalSurveyors': no_surveyors,
        'ProductivityPerSurveyor': unique_reports['DistributionPipeCoveredKm'].sum()/no_surveyors,
        'SurveyCount': surveyCount,
        'AvgSpeedKm': avg_speed_weighted.sum()/df['TotalSegments'].sum(),
        'SurveysCarDay': surveyCount/no_surveyors/day_count,
   
        'IdleTime': 100*df['IdleTimeMinutes'].sum()/df['SurveyDurationMinutes'].sum(),
        'DaysCount': day_count,
        'TotalDrivenLengthKm': df['TotalKilometers'].sum(),
        'DrivingRatio': df['TotalKilometers'].sum()/unique_reports['AssetCoveredLengthKm'].sum(),
        'NightDrivenLength': df['NightKilometers'].sum(),
        'DayDrivenLength': df['DayKilometers'].sum(),
        'NightRatio': 100*df['NightKilometers'].sum()/df['TotalKilometers'].sum(),
        'DayRatio': 100*df['DayKilometers'].sum()/df['TotalKilometers'].sum(),
    })
report_summary_agg = pd.merge(data[KPI_ReportSummary],data[KPI_SurveySummary],on='ReportId',how="left")
report_KPI_df = report_summary_agg.groupby(aggregator).apply(report_KPI_apply)
report_KPI_df = report_KPI_df.drop(columns=["DaysCount"]).round(2)

(2023, 14) 2 13
(2023, 15) 2 17
(2023, 16) 3 35
(2023, 17) 2 25
(2023, 19) 2 9
(2023, 20) 2 31
(2023, 21) 2 25
(2023, 22) 3 10
(2023, 23) 1 15
(2023, 25) 2 8
(2023, 26) 1 6
(2023, 27) 4 38
(2023, 28) 3 20
(2023, 29) 2 35
(2023, 30) 2 20
(2023, 31) 3 43
(2023, 32) 4 66
(2023, 33) 4 33
(2023, 34) 4 50
(2023, 35) 3 26
(2023, 36) 2 24
(2023, 37) 4 86
(2023, 38) 4 72
(2023, 39) 4 71
(2023, 40) 4 39
(2023, 41) 1 12
(2023, 42) 4 31
(2023, 43) 2 21
(2023, 44) 3 33
(2023, 45) 3 34
(2023, 46) 2 31
(2023, 47) 3 34
(2023, 48) 2 17
(2023, 49) 3 31
(2023, 50) 3 49
(2023, 51) 3 25
(2023, 52) 2 21
(2024, 1) 2 27
(2024, 2) 4 36
(2024, 3) 4 52
(2024, 4) 4 44
(2024, 5) 4 49
(2024, 6) 3 32
(2024, 7) 3 38
(2024, 8) 3 27
(2024, 9) 4 41
(2024, 10) 4 47
(2024, 11) 4 40
(2024, 12) 4 36
(2024, 13) 3 27
(2024, 14) 4 31
(2024, 15) 4 35
(2024, 16) 4 55
(2024, 17) 4 29
(2024, 18) 4 23
(2024, 19) 4 31
(2024, 20) 4 29
(2024, 21) 3 21
(2024, 22) 3 45
(2024, 23) 3 38
(2024, 24) 3 33
(2024, 25) 3 38
(2024, 26) 2 7
(2024

In [5]:
report_to_check = report_summary_agg[(report_summary_agg['ReportYear']==2025) & (report_summary_agg['ReportWeek']==43)]
report_to_check.head()

,ReportId,CustomerId,ReportName,ReportDate,ReportYear,ReportMonth,ReportWeek,ReportAssetLengthKm,AssetCoveredLengthKm,DistributionPipeKm,...,IdleSegments,TotalSegments,TotalKilometers,DayKilometers,NightKilometers,SegmentDurationMinutes,IdleTimeMinutes,ActiveTimeMinutes,AvgSpeedKm,LastUpdated_y
6233,67113563-8F11-DE4D-E9E3-3A1D21D7E509,BD4D080B-1D12-D329-ABD0-39FEB9804E98,CR-671135,2025-10-23 06:04:19.593000,2025,10,43,49.226880,47.833365,48.273375,...,6,310,58.687697,9.347319,49.340378,159.203067,11.770850,147.432217,29.979037,2026-06-15 09:00:33.848521
6234,67113563-8F11-DE4D-E9E3-3A1D21D7E509,BD4D080B-1D12-D329-ABD0-39FEB9804E98,CR-671135,2025-10-23 06:04:19.593000,2025,10,43,49.226880,47.833365,48.273375,...,5,300,57.053520,0.000000,57.053520,149.695400,12.927383,136.768017,29.173491,2026-06-15 09:00:33.848521
6235,67113563-8F11-DE4D-E9E3-3A1D21D7E509,BD4D080B-1D12-D329-ABD0-39FEB9804E98,CR-671135,2025-10-23 06:04:19.593000,2025,10,43,49.226880,47.833365,48.273375,...,12,301,55.824215,0.000000,55.824215,171.218900,37.676567,133.542333,27.411872,2026-06-15 09:00:33.848521
10274,AD67E8A7-2E90-8AAA-27A6-3A1D21DBD38E,BD4D080B-1D12-D329-ABD0-39FEB9804E98,CR-AD67E8,2025-10-23 06:08:37.263000,2025,10,43,43.633908,39.408141,41.546415,...,1,50,9.813470,0.000000,9.813470,36.535333,19.807000,16.728333,36.791443,2026-06-15 09:00:33.848521
10275,AD67E8A7-2E90-8AAA-27A6-3A1D21DBD38E,BD4D080B-1D12-D329-ABD0-39FEB9804E98,CR-AD67E8,2025-10-23 06:08:37.263000,2025,10,43,43.633908,39.408141,41.546415,...,8,287,55.071899,8.410374,46.661524,127.977750,27.761783,100.215967,38.111489,2026-06-15 09:00:33.848521


In [6]:
customer_id

'BD4D080B-1D12-D329-ABD0-39FEB9804E98'

In [7]:
r = report_KPI_df.reset_index()
r_long = r.melt(id_vars=aggregator, var_name='KPIId', value_name='Value')
r_long['Id'] = r_long.apply(lambda row: f"{row['KPIId']}_{customer_name}_Y{row['ReportYear']}_W{row['ReportWeek']}", axis=1)
r_long = r_long.rename(columns={'ReportWeek': 'PeriodValue', 'ReportYear': 'Year'})
r_long['PeriodType'] = "Weekly"
r_long['LastUpdated'] = datetime.now()
r_long['CustomerId'] = customer_id


In [8]:
KPI_Data.update_table(arguments = {'DataFrame': r_long, 'db_path': DB_PATH, 'PrimaryKey': 'Id'})
KPI_Data.query_table(arguments = {'db_path': DB_PATH})


,Id,KPIId,CustomerId,BoundaryRegion,Year,PeriodType,PeriodValue,Value,DataType,LastUpdated
0,FOVMain_Cadent_REastMidlands_Y2024_W16,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,East Midlands,2024,Weekly,16,None,None,2026-06-30 12:37:35.484961
1,FOVMain_Cadent_REastMidlands_Y2024_W35,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,East Midlands,2024,Weekly,35,0.95,None,2026-06-30 12:37:35.484961
2,FOVMain_Cadent_REastMidlands_Y2024_W36,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,East Midlands,2024,Weekly,36,0.96,None,2026-06-30 12:37:35.484961
3,FOVMain_Cadent_REastMidlands_Y2024_W48,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,East Midlands,2024,Weekly,48,0.94,None,2026-06-30 12:37:35.484961
4,FOVMain_Cadent_REastMidlands_Y2026_W5,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,East Midlands,2026,Weekly,5,0.97,None,2026-06-30 12:37:35.484961
...,...,...,...,...,...,...,...,...,...,...
21355,DrivingRatio_CadentRNorthLondon_Y2026_W27,DrivingRatio,BD4D080B-1D12-D329-ABD0-39FEB9804E98,North London,2026,Weekly,27,0.0,None,2026-06-30 12:37:42.372589
21356,NightDrivenLength_CadentRNorthLondon_Y2026_W27,NightDrivenLength,BD4D080B-1D12-D329-ABD0-39FEB9804E98,North London,2026,Weekly,27,0.0,None,2026-06-30 12:37:42.372589
21357,DayDrivenLength_CadentRNorthLondon_Y2026_W27,DayDrivenLength,BD4D080B-1D12-D329-ABD0-39FEB9804E98,North London,2026,Weekly,27,0.0,None,2026-06-30 12:37:42.372589
21358,NightRatio_CadentRNorthLondon_Y2026_W27,NightRatio,BD4D080B-1D12-D329-ABD0-39FEB9804E98,North London,2026,Weekly,27,0.0,None,2026-06-30 12:37:42.372589
